In [2]:
import torch
import pandas as pd
import numpy as np

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Number of GPUs:", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))

PyTorch: 2.11.0+cu128
CUDA available: True
Number of GPUs: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [3]:
!pip install -q transformers scikit-learn

In [4]:
import transformers
import sklearn

print("Transformers:", transformers.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Libraries ready!")

Transformers: 5.16.1
Scikit-learn: 1.6.1
Libraries ready!


In [5]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/datasets/potharajuneha/jarvis-json/jdft_3d-12-12-2022.json
/kaggle/input/datasets/potharajuneha/jarvis-cleaned/jarvis_cleaned.csv


In [6]:
import pandas as pd

file_path = "/kaggle/input/datasets/potharajuneha/jarvis-cleaned/jarvis_cleaned.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (75990, 17)

Columns:
['jid', 'formula', 'spg_number', 'spg_symbol', 'formation_energy_peratom', 'ehull', 'optb88vdw_bandgap', 'optb88vdw_total_energy', 'density', 'bulk_modulus_kv', 'shear_modulus_gv', 'poisson', 'epsx', 'avg_elec_mass', 'avg_hole_mass', 'slme', 'mbj_bandgap']


In [7]:
target_columns = [
    "formation_energy_peratom",
    "ehull",
    "density",
    "optb88vdw_bandgap"
]

print("Missing values:")
print(df[target_columns].isnull().sum())

print("\nNumber of unique values:")
print(df[target_columns].nunique())

Missing values:
formation_energy_peratom    0
ehull                       0
density                     0
optb88vdw_bandgap           0
dtype: int64

Number of unique values:
formation_energy_peratom    66363
ehull                       11773
density                     13855
optb88vdw_bandgap            5860
dtype: int64


In [8]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".json"):
            print(os.path.join(root, file))

/kaggle/input/datasets/potharajuneha/jarvis-json/jdft_3d-12-12-2022.json


In [9]:
import json

json_path = "/kaggle/input/datasets/potharajuneha/jarvis-json/jdft_3d-12-12-2022.json"

with open(json_path, "r") as f:
    data = json.load(f)

print("Number of records:", len(data))
print("\nFirst record:")
print(data[0])

Number of records: 75993

First record:
{'jid': 'JVASP-90856', 'spg_number': '129', 'spg_symbol': 'P4/nmm', 'formula': 'TiCuSiAs', 'formation_energy_peratom': -0.42762, 'func': 'OptB88vdW', 'optb88vdw_bandgap': 0.0, 'atoms': {'lattice_mat': [[3.566933224304235, 0.0, -0.0], [0.0, 3.566933224304235, -0.0], [-0.0, -0.0, 9.397075454186664]], 'coords': [[2.6751975000000003, 2.6751975000000003, 7.376101754328542], [0.8917325, 0.8917325, 2.0209782456714573], [0.8917325, 2.6751975000000003, 4.69854], [2.6751975000000003, 0.8917325, 4.69854], [0.8917325, 2.6751975000000003, 0.0], [2.6751975000000003, 0.8917325, 0.0], [2.6751975000000003, 2.6751975000000003, 2.8894795605846353], [0.8917325, 0.8917325, 6.507600439415366]], 'elements': ['Ti', 'Ti', 'Cu', 'Cu', 'Si', 'Si', 'As', 'As'], 'abc': [3.56693, 3.56693, 9.39708], 'angles': [90.0, 90.0, 90.0], 'cartesian': True, 'props': ['', '', '', '', '', '', '', '']}, 'slme': 'na', 'magmom_oszicar': 0.0, 'spillage': 'na', 'elastic_tensor': 'na', 'effecti

In [10]:
# Check whether JARVIS contains direct/indirect band-gap information
matches = []

for record in data:
    for key in record.keys():
        if "direct" in key.lower() or "stable" in key.lower():
            matches.append((key, record.get(key)))

print("Matching fields found:")
print(set(key for key, value in matches))


Matching fields found:
set()


In [11]:
df["is_stable"] = (df["ehull"] == 0).astype(int)

targets = [
    "formation_energy_peratom",
    "ehull",
    "density",
    "is_stable",
    "optb88vdw_bandgap"
]

print(df[targets].head())
print("\nStable:", df["is_stable"].sum())
print("Unstable:", (df["is_stable"] == 0).sum())

   formation_energy_peratom   ehull  density  is_stable  optb88vdw_bandgap
0                  -0.42762  0.0423    5.956          0              0.000
1                  -0.41596  0.0456    5.522          0              0.000
2                   0.04847  0.3183   10.960          0              0.000
3                  -0.44140  0.0000    5.145          1              0.472
4                  -0.71026  0.0156    5.718          0              0.000

Stable: 24202
Unstable: 51788


In [12]:
from sklearn.model_selection import train_test_split

# Use formula as input and the 5 properties as targets
model_df = df[
    [
        "formula",
        "formation_energy_peratom",
        "ehull",
        "density",
        "is_stable",
        "optb88vdw_bandgap"
    ]
].copy()

# 80% train, 10% validation, 10% test
train_df, temp_df = train_test_split(
    model_df,
    test_size=0.20,
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42
)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (60792, 6)
Validation: (7599, 6)
Test: (7599, 6)


In [13]:
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "alan-yahya/MatBERT"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert = AutoModel.from_pretrained(MODEL_NAME)

print("MatBERT loaded successfully!")


config.json:   0%|          | 0.00/508 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/223k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: alan-yahya/MatBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


MatBERT loaded successfully!


model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

In [14]:
import torch
from torch.utils.data import Dataset, DataLoader

class MaterialDataset(Dataset):
    def __init__(self, dataframe):
        self.formulas = dataframe["formula"].astype(str).tolist()

        self.targets = torch.tensor(
            dataframe[
                [
                    "formation_energy_peratom",
                    "ehull",
                    "density",
                    "is_stable",
                    "optb88vdw_bandgap"
                ]
            ].values,
            dtype=torch.float32
        )

        self.encodings = tokenizer(
            self.formulas,
            padding="max_length",
            truncation=True,
            max_length=32
        )

    def __len__(self):
        return len(self.formulas)

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(
                self.encodings["input_ids"][idx],
                dtype=torch.long
            ),
            "attention_mask": torch.tensor(
                self.encodings["attention_mask"][idx],
                dtype=torch.long
            ),
            "targets": self.targets[idx]
        }


train_dataset = MaterialDataset(train_df)
val_dataset = MaterialDataset(val_df)
test_dataset = MaterialDataset(test_df)

print("Datasets created successfully!")
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Datasets created successfully!
Train: 60792
Validation: 7599
Test: 7599


In [15]:
import torch
import torch.nn as nn

class MultiTaskMatBERT(nn.Module):
    def __init__(self, bert_model):
        super().__init__()

        self.bert = bert_model

        # Regression heads
        self.formation_energy_head = nn.Linear(768, 1)
        self.ehull_head = nn.Linear(768, 1)
        self.density_head = nn.Linear(768, 1)
        self.bandgap_head = nn.Linear(768, 1)

        # Classification head
        self.stable_head = nn.Linear(768, 1)

    def forward(self, input_ids, attention_mask):

        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Mean pooling
        hidden = outputs.last_hidden_state

        mask = attention_mask.unsqueeze(-1).float()

        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1)

        formation_energy = self.formation_energy_head(pooled)
        ehull = self.ehull_head(pooled)
        density = self.density_head(pooled)
        is_stable = self.stable_head(pooled)
        bandgap = self.bandgap_head(pooled)

        return (
            formation_energy,
            ehull,
            density,
            is_stable,
            bandgap
        )


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MultiTaskMatBERT(bert)
model = model.to(device)

print("Multi-task MatBERT model created successfully!")
print("Device:", device)

Multi-task MatBERT model created successfully!
Device: cuda


In [16]:
from torch.utils.data import DataLoader
import torch.nn.functional as F
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# --------------------------------------------------
# Normalize regression targets using TRAINING data
# --------------------------------------------------

regression_cols = [
    "formation_energy_peratom",
    "ehull",
    "density",
    "optb88vdw_bandgap"
]

train_means = torch.tensor(
    train_df[regression_cols].mean().values,
    dtype=torch.float32
).to(device)

train_stds = torch.tensor(
    train_df[regression_cols].std().values,
    dtype=torch.float32
).to(device)

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5
)

# Loss functions
mse_loss = nn.MSELoss()
bce_loss = nn.BCEWithLogitsLoss()

# --------------------------------------------------
# Training
# --------------------------------------------------

epochs = 3

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        targets = batch["targets"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        formation_pred = outputs[0].squeeze(-1)
        ehull_pred = outputs[1].squeeze(-1)
        density_pred = outputs[2].squeeze(-1)
        stable_pred = outputs[3].squeeze(-1)
        bandgap_pred = outputs[4].squeeze(-1)

        # Normalize regression targets
        formation_target = (
            targets[:, 0] - train_means[0]
        ) / train_stds[0]

        ehull_target = (
            targets[:, 1] - train_means[1]
        ) / train_stds[1]

        density_target = (
            targets[:, 2] - train_means[2]
        ) / train_stds[2]

        bandgap_target = (
            targets[:, 4] - train_means[3]
        ) / train_stds[3]

        # Losses
        loss_formation = mse_loss(
            formation_pred, formation_target
        )

        loss_ehull = mse_loss(
            ehull_pred, ehull_target
        )

        loss_density = mse_loss(
            density_pred, density_target
        )

        loss_stable = bce_loss(
            stable_pred, targets[:, 3]
        )

        loss_bandgap = mse_loss(
            bandgap_pred, bandgap_target
        )

        # Total multi-task loss
        loss = (
            loss_formation
            + loss_ehull
            + loss_density
            + loss_stable
            + loss_bandgap
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch + 1}/{epochs} "
        f"- Training Loss: {avg_loss:.4f}"
    )

Epoch 1/3 - Training Loss: 2.3314
Epoch 2/3 - Training Loss: 1.7152
Epoch 3/3 - Training Loss: 1.4710


In [17]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

model.eval()

all_targets = []
all_predictions = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        targets = batch["targets"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        formation_pred = outputs[0].squeeze(-1)
        ehull_pred = outputs[1].squeeze(-1)
        density_pred = outputs[2].squeeze(-1)
        stable_pred = torch.sigmoid(outputs[3].squeeze(-1))
        bandgap_pred = outputs[4].squeeze(-1)

        # Convert normalized regression predictions
        formation_pred = (
            formation_pred * train_stds[0] + train_means[0]
        )

        ehull_pred = (
            ehull_pred * train_stds[1] + train_means[1]
        )

        density_pred = (
            density_pred * train_stds[2] + train_means[2]
        )

        bandgap_pred = (
            bandgap_pred * train_stds[3] + train_means[3]
        )

        predictions = torch.stack([
            formation_pred,
            ehull_pred,
            density_pred,
            stable_pred,
            bandgap_pred
        ], dim=1)

        all_targets.append(targets.cpu())
        all_predictions.append(predictions.cpu())


all_targets = torch.cat(all_targets).numpy()
all_predictions = torch.cat(all_predictions).numpy()

# --------------------------------------------------
# Regression metrics
# --------------------------------------------------

property_names = [
    "Formation Energy",
    "Energy Above Hull",
    "Density",
    "Band Gap"
]

target_indices = [0, 1, 2, 4]

print("\n===== REGRESSION RESULTS =====")

for name, idx in zip(property_names, target_indices):

    y_true = all_targets[:, idx]
    y_pred = all_predictions[:, idx]

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(f"\n{name}")
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R²   : {r2:.4f}")


# --------------------------------------------------
# Classification metrics
# --------------------------------------------------

y_true_stable = all_targets[:, 3].astype(int)
y_pred_stable = (all_predictions[:, 3] >= 0.5).astype(int)

accuracy = accuracy_score(
    y_true_stable,
    y_pred_stable
)

precision = precision_score(
    y_true_stable,
    y_pred_stable,
    zero_division=0
)

recall = recall_score(
    y_true_stable,
    y_pred_stable,
    zero_division=0
)

f1 = f1_score(
    y_true_stable,
    y_pred_stable,
    zero_division=0
)

print("\n===== IS STABLE RESULTS =====")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")


===== REGRESSION RESULTS =====

Formation Energy
MAE  : 0.2544
RMSE : 0.3960
R²   : 0.8641

Energy Above Hull
MAE  : 0.1499
RMSE : 0.2986
R²   : 0.4996

Density
MAE  : 0.6762
RMSE : 1.0191
R²   : 0.8984

Band Gap
MAE  : 0.3918
RMSE : 0.7465
R²   : 0.6880

===== IS STABLE RESULTS =====
Accuracy : 0.7665
Precision: 0.6439
Recall   : 0.5697
F1 Score : 0.6045


In [20]:
import os

save_path = "/kaggle/working/matbert_multitask_5properties"

os.makedirs(save_path, exist_ok=True)

# Save model
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "train_means": train_means.cpu(),
        "train_stds": train_stds.cpu(),
        "target_columns": [
            "formation_energy_peratom",
            "ehull",
            "density",
            "is_stable",
            "optb88vdw_bandgap"
        ]
    },
    f"{save_path}/model.pt"
)

# Save tokenizer
tokenizer.save_pretrained(save_path)

print("Model saved successfully!")
print("Location:", save_path)

Model saved successfully!
Location: /kaggle/working/matbert_multitask_5properties


In [21]:
import pandas as pd

results_df = test_df[["formula"]].copy()

results_df["actual_formation_energy"] = all_targets[:, 0]
results_df["predicted_formation_energy"] = all_predictions[:, 0]

results_df["actual_ehull"] = all_targets[:, 1]
results_df["predicted_ehull"] = all_predictions[:, 1]

results_df["actual_density"] = all_targets[:, 2]
results_df["predicted_density"] = all_predictions[:, 2]

results_df["actual_is_stable"] = all_targets[:, 3].astype(int)
results_df["predicted_is_stable"] = (
    all_predictions[:, 3] >= 0.5
).astype(int)

results_df["actual_bandgap"] = all_targets[:, 4]
results_df["predicted_bandgap"] = all_predictions[:, 4]

results_df.to_csv(
    "/kaggle/working/matbert_5property_test_predictions.csv",
    index=False
)

print("Prediction file created successfully!")
print("Rows:", len(results_df))
print("Columns:", len(results_df.columns))

Prediction file created successfully!
Rows: 7599
Columns: 11
